In [14]:

from langchain_core.documents import Document
import os

In [15]:
os.makedirs('../data/text_files',exist_ok=True)

In [16]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}


for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding='utf-8') as f:
        f.write(content)

In [17]:
from langchain_community.document_loaders import TextLoader
loader=TextLoader("../data/text_files/python_intro.txt",encoding='utf-8')
documents=loader.load()

In [18]:
from langchain_community.document_loaders import DirectoryLoader
dirLoad=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'},
    show_progress=False
)

In [19]:
documents=dirLoad.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [20]:
import numpy as np
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import uuid
from typing import List, Tuple,Any,Dict
from sklearn.metrics.pairwise import cosine_similarity
import re

In [21]:
class EmbeddingManager:
    def __init__(self,model_name: str='all-MiniLM-L6-v2'):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            print(f"loading the embedding model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)

            print("The model loaded sucessfully")
            print("The embedding dimensions:", self.model.get_sentence_embedding_dimension())

        except Exception as e:
            print(f"There is an error in loading model {e}")
            raise
    
    def generate_embedding(self,texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("model is not found")
        print(f"Generating the embedding for {len(texts)} texts..")

        Embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"The generated embedding with shape:{Embeddings.shape}")

        return Embeddings
Embedding_Manager_obj=EmbeddingManager()
Embedding_Manager_obj

loading the embedding model: all-MiniLM-L6-v2
The model loaded sucessfully
The embedding dimensions: 384


In [22]:
print(documents)

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '), Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popul

In [23]:
def recursive_chunk(text: str, chunk_size: int=30,overlap: int=50) -> List[str]:
    def split_by(text, delimiter_regex):
        parts = re.split(delimiter_regex,text)
        merged=[]


        for i in range(0, len(parts) - 1, 2):
            merged.append(parts[i]+parts[i+1])

        if len(parts)%2 ==1:
            merged.append(parts[-1])
        return merged
    
    if len(text)<=chunk_size:
        return [text]
    
    paragraph = text.split("\n\n")
    if any(len(p)>chunk_size for p in paragraph):
        split_paragraphs=[]

        for p in paragraph:
            if len(p) <= chunk_size:
                split_paragraphs.append(p)
            else:
                sentence =split_by(p,r'(?<=[.!?])\s+')
                split_paragraphs.extend(sentence)
        paragraph = split_paragraphs
    
    refined =[]

    for segment in paragraph:
        if len(segment) <= chunk_size:
            refined.append(segment)
        else:
            words = segment.split(" ")
            temp_chunk = "" 

            for word in words:
                if len(temp_chunk)+len(word)+1 >chunk_size:
                    refined.append(temp_chunk.strip())
                    temp_chunk += word+ " "
                else:
                    temp_chunk += word + " "
            if temp_chunk:
                refined.append(temp_chunk.strip())

    #4 character level slicing +overlap

    final_chunk=[]
    for t in refined:
        start = 0
        while start < len(t):
            end = start + chunk_size
            final_chunk.append(t[start:end])
            start = end - overlap
    return final_chunk
        

In [24]:
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [25]:
chunks=recursive_chunk(documents)
print(len(chunks[0]))

2


In [26]:
class vectorStore:
    def __init__(self,collection_name: str ='pdf_documents', persist_directory: str="../data/vector_store"):

        self.collection_name = collection_name
        self.persist_directory= persist_directory
        self.client = None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            self.collection=self.client.get_or_create_collection(
                name= self.collection_name,
                metadata={"description": "PDF document embedding of the RAG"}
            )

            print(f"The vector store is initialized. Collection: {self.collection_name}")
            print(f"Existing Document is collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents:List[Any],embeddings: np.ndarray):
        if(len(documents)!=len(embeddings)):
            raise ValueError("The size of doc and embedding must be equal")
        
        print("*"*20)

        print(f"Adding {len(documents)} documents to the vector store...")

        idss=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            idss.append(doc_id)

            #prepare meta data
            metadata=dict   (doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            #add document content
            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist())


        try:
            self.collection.add(
                ids=idss,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents in the vector store")
            print(f"total documents in collection: {self.collection.count}")

        except Exception as e:
            print(f"Error in adding the document in the vector storage {e}")
            raise
vector_Store_obj=vectorStore()
vector_Store_obj

The vector store is initialized. Collection: pdf_documents
Existing Document is collection: 10


In [27]:
print(chunks)

[[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '), Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [28]:
chunks=chunks[0]

In [29]:
texts=[doc.page_content for doc in chunks]

In [30]:
texts

['Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    ',
 'Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\

In [31]:
Embeddings=Embedding_Manager_obj.generate_embedding(texts)

Generating the embedding for 2 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.39it/s]

The generated embedding with shape:(2, 384)


In [32]:
Embeddings

array([[-3.15599889e-02, -3.29403281e-02,  4.52735536e-02,
        -2.80028693e-02,  3.45180444e-02,  2.26808283e-02,
         1.27942581e-02, -6.15102127e-02, -1.22174963e-01,
        -1.72239356e-03, -6.71480969e-02,  4.71376674e-03,
         2.80751828e-02, -3.56527157e-02, -2.84922728e-03,
         5.31845307e-03,  3.91743518e-02, -1.08010219e-02,
        -9.46508162e-03, -7.42370039e-02,  4.31737602e-02,
         1.60430763e-02, -4.06411588e-02,  3.04173473e-02,
         8.15046299e-03,  5.22306710e-02,  6.50505200e-02,
         4.15119529e-02,  5.27264290e-02, -2.00100727e-02,
        -9.94696049e-04, -4.57338914e-02,  5.20974922e-04,
         5.74597940e-02, -6.97539076e-02, -1.86431184e-02,
        -2.34254487e-02,  1.54415956e-02, -1.08244922e-02,
        -5.80925448e-03, -4.65660542e-02, -4.29417789e-02,
         1.43902907e-02, -1.15238745e-02,  1.52299896e-01,
         9.40929502e-02, -9.92160067e-02, -1.05190247e-01,
        -2.30506714e-02, -1.76741034e-02, -5.79000451e-0

In [33]:
vector_Store_obj.add_documents(chunks,Embeddings)

********************
Adding 2 documents to the vector store...
Successfully added 2 documents in the vector store
total documents in collection: <bound method Collection.count of Collection(name=pdf_documents)>


In [34]:
class RAGRetrival:

    def __init__(self,vector_Store_obj:vectorStore, Embedding_Manager_obj:EmbeddingManager):
        self.vector_store=vector_Store_obj
        self.embedding_manager=Embedding_Manager_obj

    def retrieve(self, query:str, top_k: int= 5, score_threshold: float=0.0) -> List[Dict[str,Any]]:
        print(f"Retrieve document query: {query}")
        print(f"Top k: {top_k} , threshold: {score_threshold}")

        query_embedding= self.embedding_manager.generate_embedding([query][0])

        try:
            result= self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_doc=[]

            if result['documents'] and result['documents'][0]:
                documents=result['documents'][0]
                metadatas=result['metadatas'][0]
                distances=result['distances'][0]
                ids=result['ids'][0]
                
                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    #convert the distance to similarity score (chroma bd uses the cosin simialrity)
                    similarity_score=1-distance
                    if similarity_score >= score_threshold:
                        retrieved_doc.append({
                            'id': doc_id,
                            'content': document,
                            'metadata':metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank':i+1
                        })
                print(f"Retrieved {len(retrieved_doc)} document (After filtering)")

            else:

                print("No document found")
            
            return retrieved_doc
        
        except Exception as e:
            print(f"Error in retrieval: {e}")
            return[]
rag_retriever=RAGRetrival(vector_Store_obj,Embedding_Manager_obj)



In [35]:
rag_retriever

In [36]:
rag_retriever.retrieve("tell me Supervised learning")


Retrieve document query: tell me Supervised learning
Top k: 5 , threshold: 0.0
Generating the embedding for 27 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 117.57it/s]

The generated embedding with shape:(384,)
Retrieved 5 document (After filtering)


[{'id': 'doc_d7b377e9_0',
  'content': 'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    ',
  'metadata': {'doc_index': 0,
   'content_length': 575,
   'source': '..\\data\\text_files\\machine_learning.txt'},
  'similarity_score': 0.2015199065208435,
  'distance': 0.7984800934791565,
  'rank': 1},
 {'id': 'doc_fbe0c570_0',
  'content': 'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and impr

In [ ]:
api_key="REDACTED_GROQ_KEY"

In [38]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

llm=ChatGroq(api_key=api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

def reg_simple(query,rag_retriever,llm,top_k=3):
    result=rag_retriever.retrieve(query,top_k)
    context="/n/n".join([doc['content'] for doc in result]) if result else ""
    if not result:
        return "No relevant content found for this question"
    prompt=f"""Use the context to aanswer the question
        context :{context}
        question : {query}

answer: 
    """
    response=llm.invoke([prompt.format(content=context,query=query)])
    return response.content

In [39]:
query="What is machine learning"

In [40]:
response=reg_simple(query,rag_retriever,llm,3)

Retrieve document query: What is machine learning
Top k: 3 , threshold: 0.0
Generating the embedding for 24 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 122.48it/s]

The generated embedding with shape:(384,)
Retrieved 3 document (After filtering)


In [41]:
response

'Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves.'

In [42]:
def advanced_rag(query,llm,rag_retriever,top_k=5,min_score=0.2,return_context=False):
    """Rag pipeline with extra features:
     return answer, source, confidence, score, and optionally full context"""
    

    result=rag_retriever.retrieve(query,top_k,score_threshold=min_score)
    if not result:
        return {"answer":"No relevant answer found for this query",'source':[],"confidence":0.0,'context':''}
    context="\n\n".join(doc['content'] for doc in result)
    source=[{
        'source': doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page': doc['metadata'].get('page','unknown'),
        'score':doc['similarity_score'],
        'preview': doc['content'][:120]+ "..."

    } for doc in result]

    confidence = max([doc['similarity_score'] for doc in result])

    prompt=f"""Use the following context to answer the question concisely. 
    context: {context}, question: {query}, source:{source},confidance :{confidence}
"""
    response=llm.invoke([prompt])

    output  = {
        'answer': response.content,
        'source': source,
        'confidence':confidence
    }

    if return_context:
        output['context']=context
    return output
    
result= advanced_rag("Explain the concept for machine learning",llm,rag_retriever,return_context=True)

Retrieve document query: Explain the concept for machine learning
Top k: 5 , threshold: 0.2
Generating the embedding for 40 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 132.45it/s]

The generated embedding with shape:(384,)
Retrieved 5 document (After filtering)


In [43]:
result['answer']

'Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves.'

In [44]:
result['confidence']

0.44108325242996216

In [45]:
result['source']

[{'source': '..\\data\\text_files\\machine_learning.txt',
  'page': 'unknown',
  'score': 0.44108325242996216,
  'preview': 'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and impro...'},
 {'source': '..\\data\\text_files\\machine_learning.txt',
  'page': 'unknown',
  'score': 0.44108325242996216,
  'preview': 'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and impro...'},
 {'source': '..\\data\\text_files\\machine_learning.txt',
  'page': 'unknown',
  'score': 0.44108325242996216,
  'preview': 'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and impro...'},
 {'source': '..\\data\\text_files\\machine_learning.txt',
  'page': 'unknown',
  'score': 0.44108325242996216,
  'preview': 'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn 

In [46]:
result['context']

'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    \n\nMachine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns 